# Day 13(M3 Day04) 실습 4 — RAG 생성 품질 평가

**목표**: 검색이 맞았는지(검색 품질)뿐 아니라, 답변이 문서에 근거하는지·질문에 맞는지·정답과 같은지(생성 품질)를 점수로 잰다. 같은 평가 세트로 실습 1~2의 개선이 실제로 효과가 있었는지 비교한다.

| Part | 내용 |
| --- | --- |
| Part 1 | 평가 세트와 RAG 답변 함수 |
| Part 2 | 평가 지표 4가지 (Context Hit, Faithfulness, Answer Relevancy, Answer Correctness) |
| Part 3 | 전체 평가와 통과 기준 |
| Part 4 | 파이프라인 비교 — Naive(PDF) vs Markdown vs Markdown + 질의 재작성 |
| 선택 | 언어가 다른 문서 — 영어 와인 리뷰에 한국어로 묻기 |

실습 3 Part 4의 검색 정확도는 "1위 문서에 정답이 있나"만 봤다. 이번에는 **답변 자체**를 잰다.

**선수**: 실습 1을 끝까지 실행해 `ai_trend_pdf`, `ai_trend_md` 저장소를 만들어 둔다.

**데이터**: 한국지능정보사회진흥원(NIA) AI@Data Report 2025-제3호 「토픽 분석을 통한 AI 주요 트렌드 및 2026 전망」(2025.12, 22쪽). 과학기술정보통신부·NIA '초거대 AI 데이터 발굴 및 기획' 사업의 산출물이다. 보고서 규정에 따라 무단전재를 금하고, 가공·인용할 때는 출처를 「한국지능정보사회진흥원(NIA)」으로 밝힌다. 저장소에는 넣지 않고 Day13-1이 NIA 공식 주소에서 `data/ai_trend_2026.pdf`로 내려받는다.

## 0. 환경 준비

In [1]:
import os, re
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [2]:
load_dotenv()
CONN = os.getenv("DATABASE_URL").replace("postgresql://", "postgresql+psycopg://")

# PGVector마다 연결 풀을 새로 만들지 않도록 엔진 하나를 같이 쓴다 (Session pooler는 연결 15개 한도)
engine = create_engine(CONN, pool_size=2, max_overflow=0)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # temperature=0: 결과 재현성

def open_store(name):
    """Day13-1에서 만든 컬렉션을 연다. 비어 있으면 Day13-1을 먼저 실행하라고 알린다."""
    store = PGVector(embeddings=embeddings, collection_name=name, connection=engine, use_jsonb=True)
    if not store.similarity_search("AI", k=1):
        raise RuntimeError(f"'{name}' 컬렉션이 비어 있다. Day13-1 노트북을 먼저 끝까지 실행한다.")
    return store

# 출력 형식
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


# 검색결과 비교함수
def show(docs, n=120):
    for i, d in enumerate(docs, 1):
        m = d.metadata
        print(f"  [{i}] {m.get('page', '?')}쪽 #{m.get('chunk_id', '?')} | {d.page_content[:n].replace(chr(10), ' ')}")

In [3]:
pdf_store = open_store('ai_trend_pdf')
md_store = open_store('ai_trend_md')
print('저장소 준비 완료')

저장소 준비 완료


## Part 1. 평가 세트와 RAG 답변 함수

평가 문항마다 세 가지를 준비한다.

| 항목 | 쓰는 곳 |
| --- | --- |
| `question` | RAG에 던질 질문 |
| `ground_truth` | 사람이 보고서를 읽고 쓴 기준 답변 (Answer Correctness) |
| `expected_keywords` | 검색된 근거에 들어 있어야 할 핵심어 (Context Hit) |

In [4]:
eval_dataset = [
    {"question": "본 보고서의 분석 목적은 무엇인가요?",
     "ground_truth": ("본 보고서는 2025년 국내외 주요 매체에서 수집한 AI 관련 텍스트를 바탕으로 산업·기술·정책 세 영역에서 "
                      "형성되는 담론 구조와 주요 흐름을 조망하고, LDA 토픽모델링을 통해 반복적으로 등장하는 핵심 논점과 의미 축을 "
                      "도출함으로써 2026년 AI 생태계를 전망하기 위한 기초 자료를 제공하는 것을 목적으로 한다."),
     "expected_keywords": ["AI 관련 텍스트", "산업", "기술", "정책", "담론 구조", "2026년 AI 생태계"]},
    {"question": "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
     "ground_truth": ("AI 산업 분야는 AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환을 사용하고, "
                      "AI 기술 분야는 AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발을 사용하며, "
                      "AI 정책 분야는 AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책을 사용한다."),
     "expected_keywords": ["AI투자", "시장규모", "AI에이전트", "멀티모달", "AI규제", "EU AI Act"]},
    {"question": "AI 산업, AI 기술, AI 정책의 핵심 토픽은 각각 무엇인가요?",
     "ground_truth": ("AI 산업의 핵심 토픽은 시장 규모 확대와 비즈니스 구조 전환이고, AI 기술의 핵심 토픽은 기능 고도화와 "
                      "적용 범위 확장이며, AI 정책의 핵심 토픽은 안전성과 책임 중심의 규제 거버넌스이다."),
     "expected_keywords": ["시장 규모", "기능 고도화", "안전성과 책임", "규제 거버넌스"]},
    {"question": "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
     "ground_truth": ("2026년 AI 기술은 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 전망이며, 합성데이터, 추론형 AI, "
                      "멀티모달 기술이 주요 경쟁 축으로 부상하고 신뢰성 평가와 표준화의 중요성이 커질 것으로 제시된다."),
     "expected_keywords": ["지능 구조 고도화", "합성데이터", "추론형 AI", "멀티모달", "신뢰성"]},
    {"question": "AI 정책 전망에서 안전성과 책임은 왜 중요하게 다루어지나요?",
     "ground_truth": ("AI 확산이 빨라지면서 고위험 분야의 책임과 안전성 확보, 데이터·저작권 정책 명확화, 국제 규제와의 정합성이 "
                      "중요해지고 있으며, 이를 통해 예측 가능한 정책 생태계와 신뢰 기반 거버넌스를 구축할 필요가 있기 때문이다."),
     "expected_keywords": ["안전성", "책임", "고위험", "저작권", "거버넌스"]},
    {"question": "추론형 데이터는 어떤 유형으로 구분되나요?",
     "ground_truth": "추론형 데이터는 단계적 과정 추론, 근거 기반 설명, 맥락·관계 이해, 논리·취약점 검증으로 구분된다.",
     "expected_keywords": ["단계적", "근거", "맥락", "논리", "취약점"]},
]
print("평가 문항:", len(eval_dataset))

평가 문항: 6


RAG를 한 번 실행해 **답변과 검색된 근거를 함께** 돌려주는 함수를 만든다. 평가 지표 4가지는 모두 이 한 번의 결과로 계산한다. 지표마다 검색을 다시 하면 결과가 달라질 수 있기 때문이다.

In [5]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서에 근거해서만 답한다. 문서에서 확인할 수 없으면 '문서에서 확인할 수 없습니다'라고 답한다.\n\n[문서]\n{context}"),
    ("human", "{question}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()

In [10]:
# 래그 호출기
def rag_answer(question, retrieve):
    """retrieve: 질문을 받아서 Document 목록으로 반환하는 함수"""

    docs = retrieve(question) # store에서 검색 실행
    answer = answer_chain.invoke({'context':format_docs(docs), 'question':question})

    return answer, docs


In [16]:
# 래그 호출 수행
md_retrieve = md_store.as_retriever(search_kwarg={"k":3}).invoke
answer, docs = rag_answer(eval_dataset[0]['question'], md_retrieve)
print('응답 : ', answer)
print('근거 쪽 : ', [d.metadata['page'] for d in docs])

응답 :  본 보고서의 분석 목적은 산업·기술·제도 영역에서 나타나는 담론의 방향성과 구조적 특징을 종합적으로 검토하는 것입니다. 이를 위해 최근 AI 관련 국내외 주요 자료를 기반으로 하여 각 영역에서 반복적으로 등장하는 문제의식과 관심 지점을 분석하고, 현재 논의되고 있는 핵심 쟁점과 흐름을 효과적으로 확인할 수 있도록 하는 것입니다.
근거 쪽 :  [12, 9, 8, 8]


## Part 2. 평가 지표 4가지

| 지표 | 묻는 것 | 비교 대상 | 재는 방법 |
| --- | --- | --- | --- |
| Context Hit | 검색된 근거에 필요한 정보가 있나 | 근거 ↔ 핵심어 | 문자열 포함 여부 (LLM 없음) |
| Faithfulness | 답변이 근거에만 기대나 (환각이 없나) | 답변 ↔ 근거 | LLM 채점 |
| Answer Relevancy | 답변이 질문에 맞나 | 답변 ↔ 질문 | LLM 채점 |
| Answer Correctness | 답변이 기준 답변과 같은 뜻인가 | 답변 ↔ 기준 답변 | LLM 채점 |

LLM-as-a-Judge란: 사람 대신 LLM이 답변을 채점하는 방식이다. 채점 기준을 프롬프트에 적고 0.0~1.0 점수만 받는다.

### 2-1. 점수 꺼내기

"숫자만 출력하라"고 해도 모델이 `0.8점`, `Score: 0.8`처럼 답할 수 있다. 문자열에서 숫자만 꺼내고 0~1 범위로 자른다.

In [17]:
#점수 추출기
def parse_score(text):
    # TODO: text에서 첫 번째 숫자(소수 포함)를 찾으세요 (re.search)
    m = re.search(r"\d+(\.\d+)?", text)
    return max(0.0, min(float(m.group()), 1.0)) if m else 0.0

print(parse_score("0.8"), parse_score("Score: 0.75점"), parse_score("점수 없음"))

0.8 0.75 0.0


### 2-2. Context Hit — 검색 근거 확인

기대 핵심어가 근거에 2개 이상 들어 있으면 검색이 맞은 것으로 본다. 띄어쓰기·가운뎃점 차이로 놓치지 않도록 비교 전에 없앤다.

In [18]:
# TODO: expected_keywords 중 정규화한 근거(ctx)에 들어 있는 것만 모으세요
def normalize(text):
    return re.sub(r"[\s·\-_]", "", text.lower())


#expected_keywords를 정의하고 결과에 포함된 갯수를 체크해서 기준에 부합여부를 판정
def context_hit(item, docs, min_matches=2):
    ctx = normalize(format_docs(docs)) #전처리, 클리징
   
    matched = [k for k in item["expected_keywords"] if normalize(k) in ctx]
    return len(matched) >= min_matches, matched


print(context_hit(eval_dataset[0], docs))

(True, ['산업', '기술', '정책', '담론 구조'])


### 2-3. LLM이 채점하는 지표 3가지

In [26]:
# 평가 체인 생성기
def judge(system, human):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system + "\n0.0~1.0 사이 숫자로 출력한다."), 
        ("human", human)
    ])
    return prompt | llm | StrOutputParser()

# 충실
faithfulness_chain = judge("답변이 컨텍스트에 근거하는지 평가한다. 컨텍스트에 없는 내용을 만들면 낮은 점수를 준다" 
                           "근거가 없어 '문서에서 확인할 수 없습니다.'라고 답한 것은 환각이 아니라 높은 점수를 준다.",
                           "컨텍스트: {context} \n 답변: {answer}") #human-prompt > 평가하려는 리트리버의 출력 & 체인의 최종 응답
# 관련성
relevancy_chain = judge("답변이 질문에 얼마나 직접 답하는지 평가한다."
                        "핵심이 답하지 못하거나 '문서에서 확인할 수 없습니다.'로 답하면 낮은 점수를 준다.",
                        "질문: {question} \n 답변: {answer}")

# 정확함
correctness_chain = judge("생성한 답변이 기준 답변과 의미가 얼마나 같은지 평가한다."
                          "표현이 달라도 뜻이 같으면 인정하고 중요한 내용을 빠뜨리면 감점한다.",
                          "기준답변: {ground_truth} \n 생성답변 : {answer}")

In [20]:
item = eval_dataset[0]
item

{'question': '본 보고서의 분석 목적은 무엇인가요?',
 'ground_truth': '본 보고서는 2025년 국내외 주요 매체에서 수집한 AI 관련 텍스트를 바탕으로 산업·기술·정책 세 영역에서 형성되는 담론 구조와 주요 흐름을 조망하고, LDA 토픽모델링을 통해 반복적으로 등장하는 핵심 논점과 의미 축을 도출함으로써 2026년 AI 생태계를 전망하기 위한 기초 자료를 제공하는 것을 목적으로 한다.',
 'expected_keywords': ['AI 관련 텍스트', '산업', '기술', '정책', '담론 구조', '2026년 AI 생태계']}

In [ ]:
# 평가를 위해 준비한 첫번째 평가항목의 리트리버결과 & 최조응답
docs, answer

In [35]:
#평가 체인의 실행 테스트
parse_score(faithfulness_chain.invoke({"context": format_docs(docs), "answer":answer}))

1.0

In [34]:
#평가 체인의 실행 테스트
parse_score(relevancy_chain.invoke({"question": item["question"], "answer": answer}))

1.0

In [33]:
#평가 체인의 실행 테스트
parse_score(correctness_chain.invoke({"ground_truth": item["ground_truth"], "answer": answer}))

0.8

> **주의:** 세 지표는 비교 대상이 다르다. Faithfulness는 근거와, Relevancy는 질문과, Correctness는 기준 답변과 비교한다. 체인을 부를 때 넘기는 값을 바꿔 쓰면 다른 지표를 재게 된다(예: Correctness 자리에 Relevancy 체인을 부르면 기준 답변을 '질문'으로 보고 채점한다). 지표 이름과 넘기는 값을 함께 확인한다.

## Part 3. 전체 평가와 통과 기준

In [36]:
#실제 평가함수
def evaluate(item, retrieve):
    answer, docs = rag_answer(item["question"], retrieve)       # RAG는 한 번만 실행한다
    hit, matched = context_hit(item, docs)
    return {
        "question": item["question"][:28],
        "context_hit": hit,
        "faithfulness": parse_score(faithfulness_chain.invoke({"context": format_docs(docs), "answer": answer})),
        "relevancy": parse_score(relevancy_chain.invoke({"question": item["question"], "answer": answer})),
        "correctness": parse_score(correctness_chain.invoke({"ground_truth": item["ground_truth"], "answer": answer})),
    }

In [37]:
#평가 판정 함수
def judge_pass_fail(row):
    """실습용 통과 기준. 서비스에서는 도메인에 맞게 따로 정한다."""
    if not row["context_hit"]:
        return "Fail: 검색 근거 부족"
    if row["faithfulness"] < 0.7:
        return "Fail: 문서 근거 부족"
    if row["relevancy"] < 0.7:
        return "Fail: 질문과 안 맞음"
    if row["correctness"] < 0.7:
        return "Fail: 기준 답변과 다름"
    return "Pass"

In [38]:
eval_dataset

[{'question': '본 보고서의 분석 목적은 무엇인가요?',
  'ground_truth': '본 보고서는 2025년 국내외 주요 매체에서 수집한 AI 관련 텍스트를 바탕으로 산업·기술·정책 세 영역에서 형성되는 담론 구조와 주요 흐름을 조망하고, LDA 토픽모델링을 통해 반복적으로 등장하는 핵심 논점과 의미 축을 도출함으로써 2026년 AI 생태계를 전망하기 위한 기초 자료를 제공하는 것을 목적으로 한다.',
  'expected_keywords': ['AI 관련 텍스트',
   '산업',
   '기술',
   '정책',
   '담론 구조',
   '2026년 AI 생태계']},
 {'question': '산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?',
  'ground_truth': 'AI 산업 분야는 AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환을 사용하고, AI 기술 분야는 AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발을 사용하며, AI 정책 분야는 AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책을 사용한다.',
  'expected_keywords': ['AI투자',
   '시장규모',
   'AI에이전트',
   '멀티모달',
   'AI규제',
   'EU AI Act']},
 {'question': 'AI 산업, AI 기술, AI 정책의 핵심 토픽은 각각 무엇인가요?',
  'ground_truth': 'AI 산업의 핵심 토픽은 시장 규모 확대와 비즈니스 구조 전환이고, AI 기술의 핵심 토픽은 기능 고도화와 적용 범위 확장이며, AI 정책의 핵심 토픽은 안전성과 책임 중심의 규제 거버넌스이다.',
  'expected_keywords': ['시장 규모', '기능 고도화', '안전성과 책임', '규제 거버넌스']},
 {'question': '2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?',
  'ground_trut

In [43]:
import pandas as pd

df = pd.DataFrame([evaluate(item, md_retrieve) for item in eval_dataset])
df

,question,context_hit,faithfulness,relevancy,correctness
0,본 보고서의 분석 목적은 무엇인가요?,True,1.0,1.0,0.8
1,산업·기술·정책 분야별 데이터 수집 키워드는 각각,True,1.0,1.0,1.0
2,"AI 산업, AI 기술, AI 정책의 핵심 토픽은",True,1.0,0.7,0.5
3,2026년 AI 기술 전망에서 핵심 변화는 무엇인가,True,1.0,1.0,0.9
4,AI 정책 전망에서 안전성과 책임은 왜 중요하게 다,True,1.0,1.0,0.8
5,추론형 데이터는 어떤 유형으로 구분되나요?,True,1.0,1.0,0.6


In [44]:
df['판정결과'] = df.apply(judge_pass_fail, axis=1)
df

,question,context_hit,faithfulness,relevancy,correctness,판정결과
0,본 보고서의 분석 목적은 무엇인가요?,True,1.0,1.0,0.8,Pass
1,산업·기술·정책 분야별 데이터 수집 키워드는 각각,True,1.0,1.0,1.0,Pass
2,"AI 산업, AI 기술, AI 정책의 핵심 토픽은",True,1.0,0.7,0.5,Fail: 기준 답변과 다름
3,2026년 AI 기술 전망에서 핵심 변화는 무엇인가,True,1.0,1.0,0.9,Pass
4,AI 정책 전망에서 안전성과 책임은 왜 중요하게 다,True,1.0,1.0,0.8,Pass
5,추론형 데이터는 어떤 유형으로 구분되나요?,True,1.0,1.0,0.6,Fail: 기준 답변과 다름


통과 기준의 순서에 뜻이 있다. 검색이 틀리면(Context Hit 실패) 뒤의 점수는 볼 필요가 없다. 어느 단계에서 실패했는지 판정 문구로 바로 보인다.

첫 문항("본 보고서의 분석 목적은?")을 본다. 이번 실행에서 **Context Hit은 True인데 답은 "문서에서 확인할 수 없습니다"**였다.

| 지표 | 값 | 이유 |
| --- | --- | --- |
| Context Hit | True | 기대 핵심어 중 "산업", "기술", "정책", "담론 구조"가 근거에 있었다. 이 보고서 어디에나 나오는 흔한 단어다 |
| Faithfulness | 1.0 | "확인할 수 없다"는 답은 지어낸 내용이 없으므로 높은 점수 |
| Relevancy·Correctness | 0.0 | 질문에 답하지 못했다 |

핵심어가 흔한 단어면 엉뚱한 조각도 "맞았다"로 판정된다. 기대 핵심어는 **그 답에만 나오는 말**(예: "LDA 토픽모델링", "기초 자료")로 고른다. 지표 하나만 보지 않고 넷을 함께 봐야 이런 경우를 잡는다.

> **주의:** LLM 채점은 같은 입력에도 조금씩 다른 점수를 줄 수 있다. 한 번의 평가 결과로 0.05~0.1 차이를 결론 내지 않는다.

## Part 4. 파이프라인 비교

같은 평가 세트로 세 가지 RAG를 잰다.

| 이름 | 저장소 | 검색 전에 |
| --- | --- | --- |
| Naive (PDF) | `ai_trend_pdf` (실습 1 Part 1) | 없음 |
| Markdown | `ai_trend_md` (실습 1 Part 2) | 없음 |
| Markdown + 재작성 | `ai_trend_md` | 질의 재작성 (실습 2, 기준 시점 포함) |

### 관찰

이번 실행 결과:

| 파이프라인 | Context Hit | Correctness | 통과 |
| --- | --- | --- | --- |
| Naive (PDF) | 0.83 | 0.68 | 4/6 |
| Markdown | 1.00 | 0.80 | 5/6 |
| Markdown + 재작성 | 1.00 | 0.90 | 6/6 |

| 확인할 것 | 결과 |
| --- | --- |
| Markdown 변환이 검색을 개선했나 | Context Hit이 0.83에서 1.00으로 올랐다. 실습 1에서 본 표 문제가 줄었다 |
| 재작성이 도움이 됐나 | 통과가 5개에서 6개로 늘었다. 첫 문항처럼 질문 표현 때문에 검색이 빗나간 경우를 재작성이 바로잡았다 |
| 환각이 있었나 | 세 파이프라인 모두 Faithfulness가 1.0이다. 근거 밖 내용을 지어내지 않았다 |

> **참고:** 문항이 6개라 한 문항이 통과 수의 약 17%다. 차이가 한두 문항이면 평가를 다시 돌려 같은 결과가 나오는지 본 뒤 결론 낸다.

## 선택 과제 — 언어가 다른 문서: 영어 와인 리뷰에 한국어로 묻기

Day12 확장의 와인 리뷰(영어)로 BM25·의미 검색·하이브리드를 품종 기준 평가 세트로 잰다. 질문 언어가 문서와 다를 때 검색이 어떻게 되는지 본다. 와인 데이터는 Day12 확장 노트북이 내려받은 `data/wine-reviews.zip`을 쓰고, 없으면 새로 받는다.

In [ ]:
import urllib.request, zipfile
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

### 와인 리뷰로 반복 — 한국어 질문과 영어 문서

실습 1~4는 한국어 문서와 한국어 질문이었다. Day12 확장의 와인 리뷰는 **영어**다. 질문 언어가 문서와 다를 때 BM25·의미 검색·하이브리드가 각각 어떻게 되는지 평가 세트로 잰다.

> **참고:** Day12 확장 노트북을 실행했다면 `wine_reviews` 컬렉션을 그대로 쓴다. 없으면 아래 셀이 같은 표본(3,000건, `random_state=42`)으로 다시 만든다.

In [ ]:
import urllib.request, zipfile
import pandas as pd

ZIP = Path("data/wine-reviews.zip")
if not ZIP.exists():
    ZIP.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://www.kaggle.com/api/v1/datasets/download/zynicide/wine-reviews", ZIP)
with zipfile.ZipFile(ZIP) as z:
    wine_df = pd.read_csv(z.open("winemag-data-130k-v2.csv"), index_col=0)
sample = wine_df.dropna(subset=["price", "country", "variety"]).sample(3000, random_state=42)
wine_docs = [Document(page_content=r.description,
                      metadata={"title": r.title, "variety": r.variety, "country": r.country,
                                "province": r.province if pd.notna(r.province) else "",
                                "points": int(r.points), "price": float(r.price)})
             for r in sample.itertuples()]

wine_store = PGVector(embeddings=embeddings, collection_name="wine_reviews", connection=engine, use_jsonb=True)
if not wine_store.similarity_search("wine", k=1):
    for i in range(0, len(wine_docs), 500):
        wine_store.add_documents(wine_docs[i:i + 500])
    print("wine_reviews 다시 만듦")
print("와인 문서:", len(wine_docs))

#### 5-1. 세 가지 리트리버(Retriever)

BM25는 같은 3,000건으로 메모리에 만든다. 리뷰가 영어이므로 토크나이저는 영어 단어만 소문자로 뽑는다.

In [ ]:
def tok_en(text):
    # TODO: text를 소문자로 바꾼 뒤 영어 단어([a-z]+)만 목록으로 뽑아 반환하세요
    return []

wine_bm25 = BM25Retriever.from_documents(wine_docs, preprocess_func=tok_en)
wine_bm25.k = 5
wine_dense = wine_store.as_retriever(search_kwargs={"k": 5})
wine_hybrid = EnsembleRetriever(retrievers=[wine_bm25, wine_dense], weights=[0.5, 0.5])

print(tok_en("crisp white wine with citrus"))
print(tok_en("시트러스 향이 나는 상쾌한 화이트 와인"))

한국어 질문을 이 토크나이저에 넣으면 빈 목록이 나온다. BM25가 비교할 단어가 하나도 없다는 뜻이다.

#### 5-2. 같은 뜻, 다른 언어

같은 질문을 영어와 한국어로 물어 품종을 비교한다.

In [ ]:
def varieties(retriever, q):
    return [d.metadata["variety"] for d in retriever.invoke(q)[:5]]

pair = ("crisp, zesty white wine with citrus and high acidity",
        "산미가 높고 레몬·자몽 같은 시트러스 향이 나는 상쾌한 화이트 와인")
for q in pair:
    print("Q:", q)
    print("  BM25  :", varieties(wine_bm25, q))
    print("  의미   :", varieties(wine_dense, q))
    print("  하이브리드:", varieties(wine_hybrid, q))

#### 5-3. 메타데이터로 정답을 정한 평가 세트

사람이 문서마다 정답을 달지 않아도, **품종**(`variety`)으로 정답을 정할 수 있다. "상큼한 화이트"를 물었으면 상위 5개 중 산미 높은 화이트 품종이 몇 개인지 센다(Precision@5).

In [ ]:
WINE_EVAL = [
    ("crisp, zesty white wine with citrus and high acidity",
     "산미가 높고 레몬·자몽 같은 시트러스 향이 나는 상쾌한 화이트 와인",
     {"Sauvignon Blanc", "Riesling", "Pinot Grigio", "Pinot Gris", "Albariño", "Grüner Veltliner",
      "Vermentino", "Chenin Blanc", "Portuguese White"}),
    ("full-bodied red with firm tannins and dark fruit like blackberry and cassis",
     "블랙베리·카시스 같은 검은 과일 향에 타닌이 단단한 묵직한 레드 와인",
     {"Cabernet Sauvignon", "Bordeaux-style Red Blend", "Malbec", "Syrah", "Shiraz", "Petite Sirah",
      "Nebbiolo", "Tannat", "Cabernet Franc", "Red Blend", "Merlot", "Sangiovese Grosso", "Portuguese Red", "Tempranillo"}),
    ("sparkling wine with fine bubbles and toasty brioche notes",
     "고운 기포와 구운 빵(브리오슈) 향이 나는 스파클링 와인",
     {"Sparkling Blend", "Champagne Blend", "Glera", "Portuguese Sparkling"}),
    ("light-bodied red with bright red cherry, raspberry and earthy notes",
     "체리·라즈베리 같은 붉은 과일 향에 흙내음이 나는 가벼운 레드 와인",
     {"Pinot Noir", "Gamay", "Barbera", "Sangiovese", "Grenache"}),
]

def precision_at_5(retriever, q, good):
    # TODO: 상위 5개 문서 중 metadata["variety"]가 good에 들어 있는 비율을 반환하세요
    return 0.0

rows = []
for name, r in [("BM25", wine_bm25), ("의미", wine_dense), ("하이브리드", wine_hybrid)]:
    en = sum(precision_at_5(r, e, g) for e, _, g in WINE_EVAL) / len(WINE_EVAL)
    ko = sum(precision_at_5(r, k, g) for _, k, g in WINE_EVAL) / len(WINE_EVAL)
    rows.append((name, en, ko))

print(f"{'검색기':<10}{'영어 질문':>10}{'한국어 질문':>12}")
for name, en, ko in rows:
    print(f"{name:<10}{en:>10.2f}{ko:>12.2f}")

### 관찰

출력 표에서 세 가지를 확인한다.

| 확인할 것 | 보이는 것 |
| --- | --- |
| 영어 질문 vs 한국어 질문 | 한국어로 물으면 세 리트리버 모두 점수가 떨어진다. 질문과 문서의 언어가 다르면 검색 자체가 약해진다 |
| 한국어 질문의 BM25 | 비교할 단어가 없어서(5-1의 빈 목록) 맞는 품종이 우연히 섞인 만큼만 나온다 |
| 하이브리드 vs 의미 검색 | 하이브리드가 의미 검색보다 낫지 않다. 두 리트리버 중 하나가 약하면 섞어도 이득이 없다 |

> **주의:** 이 평가 세트는 질문 4개 × 상위 5개, 모두 20칸이다. 한 칸이 0.05이므로 0.05~0.10 차이는 우연일 수 있다. 리트리버를 고를 때는 질문 수를 늘린 평가 세트로 다시 잰다. 하이브리드가 항상 낫다고 가정하지 않고, **평가 세트로 재 보고** 정한다. 실습 3 Part 4와 이 노트북 Part 3에서 한 일이 바로 이것이다.

> **참고 (선택 과제):** Day12 확장의 질의 변환처럼 한국어 질문을 LLM으로 영어 테이스팅 노트 문장으로 바꿔 검색하면 점수가 달라지는지 직접 재 본다. `WINE_EVAL`의 한국어 질문을 바꾼 문장으로 `precision_at_5`를 계산하면 된다.

## 확인 문제

1. 평가 지표 4가지는 각각 무엇과 무엇을 비교하는가?
2. 지표마다 검색을 다시 하지 않고 한 번의 RAG 결과로 계산하는 이유는?
3. Context Hit은 LLM 없이 재는데, 나머지 셋은 왜 LLM이 채점하는가? LLM 채점의 약점은?
4. 통과 기준에서 Context Hit을 가장 먼저 보는 이유는?
5. Part 4 결과로 볼 때 Markdown 변환과 질의 재작성은 각각 효과가 있었나? 문항 6개로 결론 내릴 때 조심할 점은?